In [ ]:
# ruff: noqa: I001
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import inspect
import importlib.metadata as importlib_metadata
import subprocess
import sys
import re
import time
from pathlib import Path

_START_TIME = time.time()
BOOTSTRAP_PACKAGES = ["pyyaml==6.0.3", "huggingface_hub==1.13.0"]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *BOOTSTRAP_PACKAGES])

import yaml
from huggingface_hub import HfApi, hf_hub_download


def resolve_hf_token():
    token = os.environ.get("HF_TOKEN") or os.environ.get("HUGGINGFACE_TOKEN")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient

        secrets = UserSecretsClient()
        return secrets.get_secret("HF_TOKEN")
    except Exception:
        return None


def require_config(config):
    required_sections = [
        "environment",
        "repos",
        "model",
        "lora",
        "training",
        "collection",
        "evaluation",
        "data_sources",
        "release",
    ]
    missing = [section for section in required_sections if section not in config]
    if missing:
        raise RuntimeError(f"SFT config missing sections: {missing}")
    packages = config["environment"].get("pip_packages", [])
    if not packages or any("==" not in package for package in packages):
        raise RuntimeError("Kaggle pip packages must be exact pins in sft_05b.yaml.")
    torch_packages = config["environment"].get("torch_pip_packages", [])
    if torch_packages and any("==" not in package for package in torch_packages):
        raise RuntimeError("Kaggle torch packages must be exact pins in sft_05b.yaml.")
    train_cfg = config["training"]
    if train_cfg.get("fp16") is not True or train_cfg.get("bf16") is not False:
        raise RuntimeError("Kaggle remote GPU config must use fp16=True and bf16=False.")
    expected_remote_training = {
        "per_device_train_batch_size": 1,
        "gradient_accumulation_steps": 16,
        "max_seq_length": 1024,
        "loss_type": "chunked_nll",
    }
    mismatches = {
        key: train_cfg.get(key)
        for key, expected in expected_remote_training.items()
        if train_cfg.get(key) != expected
    }
    if mismatches:
        raise RuntimeError(
            "Downloaded SFT YAML is stale for this remote run. "
            f"Expected {expected_remote_training}, got {mismatches}. "
            "Republish the dataset handoff before starting Kaggle."
        )
    print("Loaded remote training memory settings:")
    for key in expected_remote_training:
        print(f"  {key}: {train_cfg.get(key)}")


def install_torch_stack(cfg):
    torch_packages = cfg["environment"].get("torch_pip_packages", [])
    if not torch_packages:
        return None
    command = [sys.executable, "-m", "pip", "install", "-q", "--upgrade"]
    torch_index_url = cfg["environment"].get("torch_index_url")
    if torch_index_url:
        command.extend(["--index-url", torch_index_url])
    command.extend(torch_packages)
    subprocess.check_call(command)
    constraints_path = Path("/kaggle/working/dataforge-pip-constraints.txt")
    constraints_path.write_text("\n".join(torch_packages) + "\n", encoding="utf-8")
    return constraints_path


def uninstall_remote_conflicts(cfg):
    packages = list(cfg["environment"].get("uninstall_packages", []))
    if packages:
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", *packages], check=False)


HF_TOKEN = resolve_hf_token()
if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN Kaggle secret is required for this release notebook. "
        "Add the secret in Kaggle, attach it to the notebook, restart the session, and rerun from the top."
    )
os.environ["HF_TOKEN"] = HF_TOKEN
os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
os.environ["HUGGINGFACE_HUB_TOKEN"] = HF_TOKEN

api = HfApi(token=HF_TOKEN)
whoami = api.whoami(token=HF_TOKEN)
HF_USER = whoami.get("name")
if not isinstance(HF_USER, str) or not HF_USER:
    raise RuntimeError("Could not resolve Hugging Face username from HF_TOKEN.")
DATASET_REPO = f"{HF_USER}/dataforge-sft-trajectories"
DATASET_INFO = api.repo_info(DATASET_REPO, repo_type="dataset", token=HF_TOKEN)
DATASET_SHA = DATASET_INFO.sha or "unknown"
CONFIG_FILENAME = os.environ.get("DATAFORGE_SFT_CONFIG", "sft_05b_v4.yaml")
CONFIG_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO, filename=CONFIG_FILENAME, repo_type="dataset", token=HF_TOKEN
    )
)
config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
require_config(config)

uninstall_remote_conflicts(config)
constraints_path = install_torch_stack(config)
install_command = [sys.executable, "-m", "pip", "install", "-q", "--upgrade"]
if constraints_path is not None:
    install_command.extend(["--constraint", str(constraints_path)])
install_command.extend(config["environment"]["pip_packages"])
subprocess.check_call(install_command)
uninstall_remote_conflicts(config)
trl_version = importlib_metadata.version("trl")
print("trl:", trl_version)
from trl.trainer.sft_trainer import SFTTrainer as _SFTTrainer

if config["training"].get("loss_type") == "chunked_nll" and "chunked_nll" not in inspect.getsource(_SFTTrainer):
    raise RuntimeError(
        "Installed TRL does not support loss_type='chunked_nll'. "
        "Check the sft_05b.yaml trl==1.4.0 pin, restart Kaggle, and rerun from the top."
    )
try:
    print("torchao:", importlib_metadata.version("torchao"))
except importlib_metadata.PackageNotFoundError:
    print("torchao: not installed")
MODEL_REPO = config["repos"]["model_repo_template"].format(hf_user=HF_USER)
print(f"Resolved HF dataset repo: {DATASET_REPO}")
print(f"Resolved HF dataset SHA: {DATASET_SHA}")
print(f"Resolved HF model repo: {MODEL_REPO}")


In [ ]:
import hashlib
import json

from datasets import load_dataset

DATA_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=config["repos"]["trajectory_filename"],
        repo_type="dataset",
        token=HF_TOKEN,
    )
)
CARD_TEMPLATE_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=config["repos"]["model_card_template_filename"],
        repo_type="dataset",
        token=HF_TOKEN,
    )
)
SPLIT_MANIFEST_PATH = Path(
    hf_hub_download(
        repo_id=DATASET_REPO,
        filename=config["repos"].get("split_manifest_filename", "split_manifest.json"),
        repo_type="dataset",
        token=HF_TOKEN,
    )
)


def file_sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


def assert_manifest_has_no_label_fields(value, path="manifest"):
    forbidden = {
        "clean",
        "clean_value",
        "ground_truth",
        "new_value",
        "normalization_candidates",
        "repairs",
        "suggested_value",
    }
    if isinstance(value, dict):
        for key, child in value.items():
            if str(key) in forbidden:
                raise RuntimeError(f"split_manifest.json leaks label-bearing field {path}.{key}")
            assert_manifest_has_no_label_fields(child, f"{path}.{key}")
    elif isinstance(value, list):
        for index, child in enumerate(value):
            assert_manifest_has_no_label_fields(child, f"{path}[{index}]")


def validate_split_manifest(manifest, cfg):
    assert_manifest_has_no_label_fields(manifest)
    if manifest.get("schema_version") != "split_manifest_v1":
        raise RuntimeError("split_manifest.json schema_version must be split_manifest_v1")
    if manifest.get("collection_method") != "oracle_from_clean_diff":
        raise RuntimeError("split_manifest.json collection_method must be oracle_from_clean_diff")
    datasets = manifest.get("datasets")
    if not isinstance(datasets, list) or not datasets:
        raise RuntimeError("split_manifest.json must include a non-empty datasets list")
    configured = set(cfg["collection"].get("datasets", []))
    seen = set()
    for item in datasets:
        dataset = item.get("dataset")
        if dataset in seen:
            raise RuntimeError(f"split_manifest.json duplicates dataset {dataset!r}")
        seen.add(dataset)
        train = item.get("train", [])
        eval_rows = item.get("eval", [])
        if item.get("train_rows") != len(train) or item.get("eval_rows") != len(eval_rows):
            raise RuntimeError(f"split_manifest.json row counts disagree for {dataset!r}")
        for split_name, rows in (("train", train), ("eval", eval_rows)):
            row_ids = set()
            for row in rows:
                digest = row.get("dirty_row_sha256")
                if not isinstance(row.get("row"), int) or not isinstance(digest, str) or len(digest) != 64:
                    raise RuntimeError(f"split_manifest.json has malformed {dataset}.{split_name} row entry")
                if row["row"] in row_ids:
                    raise RuntimeError(f"split_manifest.json duplicates {dataset}.{split_name} row {row['row']}")
                row_ids.add(row["row"])
    missing = configured - seen
    if missing:
        raise RuntimeError(f"split_manifest.json missing configured datasets: {sorted(missing)}")


def validate_trajectory_dataset(dataset, cfg):
    record_count = len(dataset)
    if record_count < 2:
        raise RuntimeError(
            f"Need at least 2 SFT records for a non-empty train/held-out split; found {record_count}."
        )
    expected_schema = cfg["collection"]["schema_version"]
    expected_teacher = cfg["collection"]["teacher_model"]
    expected_provider = cfg["collection"].get("teacher_provider")
    allowed_methods = set(cfg["collection"].get("collection_methods", ["llm_react_chunk"]))
    oracle_cfg = cfg["collection"].get("oracle", {})
    oracle_teacher = (oracle_cfg.get("provider", "oracle"), oracle_cfg.get("model", "clean-diff-v1"))
    accepted_teachers = {(expected_provider, expected_teacher)}
    for item in cfg["collection"].get("accepted_teachers", []):
        accepted_teachers.add((item.get("provider"), item.get("model")))
    min_episode_f1 = float(cfg["collection"]["min_episode_f1"])
    seen_ids = set()
    seen_keys = set()
    for index, record in enumerate(dataset, start=1):
        if record.get("schema_version") != expected_schema:
            raise RuntimeError(f"Record {index} has wrong schema_version: {record.get('schema_version')!r}")
        trajectory_id = record.get("trajectory_id")
        if trajectory_id in seen_ids:
            raise RuntimeError(f"Duplicate trajectory_id in HF dataset: {trajectory_id}")
        seen_ids.add(trajectory_id)
        key = (record.get("task_id"), int(record.get("seed", -1)), int(record.get("chunk_index", -1)))
        if key in seen_keys:
            raise RuntimeError(f"Duplicate SFT chunk key in HF dataset: {key}")
        seen_keys.add(key)
        metrics = record.get("metrics") or {}
        if float(metrics.get("episode_f1", -1.0)) < min_episode_f1:
            raise RuntimeError(f"Record {index} is below the configured episode F1 floor.")
        provenance = record.get("provenance") or {}
        collection_method = provenance.get("collection_method", "llm_react_chunk")
        if collection_method not in allowed_methods:
            raise RuntimeError(f"Record {index} collection method does not match sft_05b.yaml.")
        teacher = record.get("teacher") or {}
        teacher_key = (teacher.get("provider"), teacher.get("model"))
        if collection_method == "oracle_from_clean_diff":
            if teacher_key != oracle_teacher:
                raise RuntimeError(f"Record {index} oracle metadata does not match sft_05b.yaml.")
            state = record.get("state") or {}
            if state.get("normalization_candidates"):
                raise RuntimeError(f"Record {index} leaks oracle labels through normalization_candidates.")
        elif teacher_key not in accepted_teachers:
            raise RuntimeError(f"Record {index} teacher metadata does not match sft_05b.yaml.")
        if not record.get("messages"):
            raise RuntimeError(f"Record {index} has no chat messages.")
    test_size = min(100, max(1, record_count // 10))
    if record_count - test_size < 1:
        raise RuntimeError("SFT split would leave no training records.")
    return test_size


split_manifest = json.loads(SPLIT_MANIFEST_PATH.read_text(encoding="utf-8"))
validate_split_manifest(split_manifest, config)
raw_dataset = load_dataset("json", data_files=str(DATA_PATH), split="train")
test_size = validate_trajectory_dataset(raw_dataset, config)
split = raw_dataset.train_test_split(test_size=test_size, seed=42)
train_records = split["train"]
heldout_records = split["test"]
print(
    f"Loaded {len(train_records)} training records and {len(heldout_records)} local holdout records "
    f"from {config['repos']['trajectory_filename']}"
)


In [ ]:
import torch
from peft import LoraConfig
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if not torch.cuda.is_available():
    raise RuntimeError("Kaggle GPU runtime is required; enable a T4 or compatible CUDA GPU.")
gpu_name = torch.cuda.get_device_name(0)
gpu_capability = torch.cuda.get_device_capability(0)
torch_arches = set(torch.cuda.get_arch_list())
print(f"CUDA GPU: {gpu_name}")
print(f"CUDA capability: sm_{gpu_capability[0]}{gpu_capability[1]}")
print(f"torch: {torch.__version__}")
print(f"torch CUDA: {torch.version.cuda}")
print("torch arch list:", " ".join(torch.cuda.get_arch_list()))
if gpu_capability[0] == 6 and "sm_60" not in torch_arches:
    raise RuntimeError(
        "Kaggle assigned a Pascal/P100 GPU, but the installed PyTorch wheel lacks sm_60 kernels. "
        "Use the sft_05b_v2 torch_index_url/torch_pip_packages pins and rerun from a clean Kaggle session."
    )

quant_cfg = config["model"]["quantization"]
bnb_config = BitsAndBytesConfig(
    load_in_4bit=quant_cfg["load_in_4bit"],
    bnb_4bit_quant_type=quant_cfg["bnb_4bit_quant_type"],
    bnb_4bit_use_double_quant=quant_cfg["bnb_4bit_use_double_quant"],
    bnb_4bit_compute_dtype=torch.float16,
)

base_model_id = config["model"]["base_model"]
tokenizer = AutoTokenizer.from_pretrained(
    base_model_id, token=HF_TOKEN, trust_remote_code=config["model"]["trust_remote_code"]
)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    dtype=torch.float16,
    device_map={"": 0},
    token=HF_TOKEN,
    trust_remote_code=config["model"]["trust_remote_code"],
)
model.config.use_cache = False

lora = config["lora"]
peft_config = LoraConfig(
    r=lora["r"],
    lora_alpha=lora["alpha"],
    lora_dropout=lora["dropout"],
    bias=lora["bias"],
    task_type=lora["task_type"],
    target_modules=lora["target_modules"],
)


In [ ]:
import gc
from collections import Counter

from trl import SFTConfig, SFTTrainer


def render_chat_text(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["messages"], tokenize=False, add_generation_prompt=False
        )
    }


train_dataset = train_records.map(render_chat_text, remove_columns=train_records.column_names)


def latest_checkpoint_path(output_dir: Path):
    checkpoints = sorted(
        output_dir.glob("checkpoint-*"), key=lambda path: int(path.name.split("-")[-1])
    )
    return str(checkpoints[-1]) if checkpoints else None


train_cfg = config["training"]
output_dir = Path(train_cfg["output_dir"])
training_args = SFTConfig(
    output_dir=str(output_dir),
    num_train_epochs=train_cfg["num_train_epochs"],
    per_device_train_batch_size=train_cfg["per_device_train_batch_size"],
    gradient_accumulation_steps=train_cfg["gradient_accumulation_steps"],
    learning_rate=train_cfg["learning_rate"],
    lr_scheduler_type=train_cfg["lr_scheduler_type"],
    warmup_ratio=train_cfg["warmup_ratio"],
    weight_decay=train_cfg["weight_decay"],
    max_length=train_cfg["max_seq_length"],
    packing=train_cfg["packing"],
    logging_steps=train_cfg["logging_steps"],
    save_steps=train_cfg["save_steps"],
    save_total_limit=train_cfg["save_total_limit"],
    fp16=train_cfg["fp16"],
    bf16=train_cfg["bf16"],
    gradient_checkpointing=train_cfg["gradient_checkpointing"],
    report_to=train_cfg["report_to"],
    dataset_text_field="text",
    loss_type=train_cfg["loss_type"],
)
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    peft_config=peft_config,
    processing_class=tokenizer,
)
for _name, param in trainer.model.named_parameters():
    if param.requires_grad and param.dtype != torch.float32:
        param.data = param.data.float()

trainable_dtypes = Counter(
    str(param.dtype) for param in trainer.model.parameters() if param.requires_grad
)
print("Trainable parameter dtypes:", trainable_dtypes)

if "torch.bfloat16" in trainable_dtypes:
    raise RuntimeError("bf16 trainable parameters remain; T4 must train with fp16/bf16=False.")

latest_checkpoint = None
print("Starting fresh T4 fp16 run; old checkpoints are ignored to avoid stale bf16 adapter state.")
trainer.train(resume_from_checkpoint=latest_checkpoint)
trainer.save_model(train_cfg["adapter_dir"])
tokenizer.save_pretrained(train_cfg["adapter_dir"])
del trainer, model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
from peft import PeftModel

adapter_dir = Path(config["training"]["adapter_dir"])
merged_dir = Path(config["training"]["merged_dir"])
if not adapter_dir.exists():
    raise RuntimeError(f"Missing trained adapter directory: {adapter_dir}")
base_for_merge = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    dtype=torch.float16,
    device_map="auto",
    token=HF_TOKEN,
    trust_remote_code=config["model"]["trust_remote_code"],
)
merged = PeftModel.from_pretrained(base_for_merge, adapter_dir)
merged = merged.merge_and_unload()
merged.save_pretrained(merged_dir, safe_serialization=True)
tokenizer.save_pretrained(merged_dir)
del merged, base_for_merge
gc.collect()
torch.cuda.empty_cache()
print(f"Merged LoRA adapter into {merged_dir}")


In [ ]:
import time
from collections import Counter, OrderedDict

import pandas as pd

VALID_INFERABILITY_LABELS = {
    "deterministic_normalization",
    "context_derivable",
    "external_reference_required",
    "not_inferable_from_prompt",
}
PROMOTION_SLICE = config.get("evaluation", {}).get("promotion_slice", "deterministic_normalization")

SYSTEM_PROMPT = (
    "You repair tabular data by proposing exact cell replacements. "
    "Rows must be absolute row ids from valid_rows and columns must exactly match one of "
    "the allowed_columns values. "
    "Use only the provided dirty target rows and optional context rows. "
    "Return strict JSON only in this object shape: "
    '{"action":"submit_repairs","repairs":[{"row":0,"column":"Column",'
    '"new_value":"value","reason":"why"}]}. '
    'Use {"action":"finish","repairs":[]} when no cells should be changed. '
    "Do not wrap the JSON in markdown code fences."
)


def load_bench_dataset(name):
    source = config["data_sources"][name]
    dirty = pd.read_csv(source["dirty_url"], dtype=str, keep_default_na=False, na_filter=False)
    clean = pd.read_csv(source["clean_url"], dtype=str, keep_default_na=False, na_filter=False)
    if len(dirty.index) != len(clean.index) or len(dirty.columns) != len(clean.columns):
        raise RuntimeError(f"Dirty/clean shape mismatch for held-out dataset {name}.")
    dirty.columns = [str(column) for column in clean.columns]
    ground_truth = []
    for row_idx, (dirty_row, clean_row) in enumerate(
        zip(dirty.itertuples(index=False, name=None), clean.itertuples(index=False, name=None), strict=True)
    ):
        for column, dirty_value, clean_value in zip(clean.columns, dirty_row, clean_row, strict=True):
            if str(dirty_value) != str(clean_value):
                ground_truth.append({"row": row_idx, "column": str(column), "clean_value": str(clean_value)})
    return dirty, tuple(str(column) for column in clean.columns), ground_truth


def chunk_rows(df, columns, truth, seed, width=None, *, prefer_truth=True):
    if width is None:
        width = int(config["evaluation"].get("chunk_width", 4))
    width = min(width, len(df.index))
    if width <= 0:
        return []
    digest = hashlib.sha256(str(seed).encode("utf-8")).hexdigest()
    truth_rows = {int(cell["row"]) for cell in truth}
    if truth and prefer_truth:
        anchor = int(truth[int(digest[:8], 16) % len(truth)]["row"])
        start = min(max(anchor - width // 2, 0), len(df.index) - width)
    else:
        max_start = max(0, len(df.index) - width)
        clean_starts = [
            start
            for start in range(max_start + 1)
            if not truth_rows.intersection(range(start, start + width))
        ]
        candidates = clean_starts if clean_starts else list(range(max_start + 1))
        start = candidates[int(digest[:8], 16) % len(candidates)]
    rows = []
    for row_idx in range(start, start + width):
        row = {"_row": str(row_idx)}
        for column in columns:
            row[column] = str(df.iloc[row_idx][column])
        rows.append(row)
    return rows


_NUMBER_RE = re.compile(r"[-+]?\d+(?:\.\d+)?")
_TIME_RE = re.compile(r"\d{1,2}:\d{2}\s*(?:a\.m\.|p\.m\.)", re.IGNORECASE)
_FLIGHT_STATUS_RE = re.compile(
    r"\s+(?:on\s+time|delayed|cancelled|canceled|arrived|departed|early|late)\b.*$",
    re.IGNORECASE,
)
_FLIGHT_DATE_TOKEN_RE = re.compile(
    r"\b(?:mon|tue|wed|thu|fri|sat|sun|jan|feb|mar|apr|may|jun|jul|aug|sep|sept|oct|nov|dec)\b"
    r"|\d{1,2}[-/][A-Za-z]{3}[-/]\d{2,4}"
    r"|[A-Za-z]{3}\s+\d{1,2},?\s+\d{2,4}",
    re.IGNORECASE,
)
_FLIGHT_TIME_COLUMNS = {"sched_dep_time", "act_dep_time", "sched_arr_time", "act_arr_time"}


def _normalize_number_text(value):
    number = float(value)
    if number.is_integer():
        return str(int(number))
    return str(number).rstrip("0").rstrip(".")


def deterministic_candidate(dataset_name, column, raw_value):
    value = str(raw_value).strip()
    if dataset_name == "beers":
        if column == "ounces":
            match = _NUMBER_RE.search(value)
            if match and re.search(r"\boz\.?\b|\bounce\b", value, re.IGNORECASE):
                return _normalize_number_text(match.group(0))
        if column == "abv" and value.endswith("%"):
            return value.rstrip("%").strip()
        if column in {"ibu", "abv"} and value.upper() in {"N/A", "NA", "NULL", "NONE"}:
            return ""
    if dataset_name == "flights" and column in _FLIGHT_TIME_COLUMNS:
        if not value or "(" in value or ")" in value:
            return None
        without_status = _FLIGHT_STATUS_RE.sub("", value).strip()
        if without_status != value and _TIME_RE.fullmatch(without_status):
            return without_status
        if _FLIGHT_DATE_TOKEN_RE.search(value):
            time_matches = _TIME_RE.findall(value)
            if len(time_matches) == 1:
                return str(time_matches[0]).strip()
    return None


def inferability_for_eval_task(dataset_name, rows, task_truth):
    if not task_truth:
        return "not_inferable_from_prompt"
    rows_by_id = {int(row["_row"]): row for row in rows}
    deterministic = True
    for cell in task_truth:
        row = rows_by_id.get(int(cell["row"]))
        if row is None:
            deterministic = False
            break
        candidate = deterministic_candidate(dataset_name, str(cell["column"]), row.get(str(cell["column"]), ""))
        if candidate != str(cell["clean_value"]):
            deterministic = False
            break
    return "deterministic_normalization" if deterministic else "external_reference_required"


def parse_json_payload(text):
    text = text.strip()
    if text.startswith("```"):
        fence_lines = text.splitlines()
        if len(fence_lines) >= 3:
            text = "\n".join(fence_lines[1:-1]).strip()
    decoder = json.JSONDecoder()
    saw_start = False
    for offset, char in enumerate(text):
        if char not in "[{":
            continue
        saw_start = True
        try:
            payload, _ = decoder.raw_decode(text[offset:])
        except json.JSONDecodeError:
            continue
        if isinstance(payload, (dict, list)):
            return payload, None
    return None, "truncated_json" if saw_start else "parse_failure"


def parse_repairs(text):
    payload, error_kind = parse_json_payload(text)
    if isinstance(payload, dict):
        repairs = payload.get("repairs", [])
    elif isinstance(payload, list):
        repairs = payload
    else:
        repairs = []
    if not isinstance(repairs, list):
        return [], "schema_error"
    parsed = []
    for repair in repairs:
        if not isinstance(repair, dict):
            continue
        if not {"row", "column", "new_value"}.issubset(repair):
            continue
        try:
            row = int(repair["row"])
        except (TypeError, ValueError):
            continue
        parsed.append(
            {
                "row": row,
                "column": str(repair["column"]),
                "new_value": str(repair["new_value"]),
                "reason": str(repair.get("reason", "repair proposal")),
            }
        )
    return parsed, error_kind


def score_repairs(truth, repairs):
    truth_map = {(int(cell["row"]), str(cell["column"])): str(cell["clean_value"]) for cell in truth}
    predictions = OrderedDict()
    for repair in repairs:
        key = (int(repair["row"]), str(repair["column"]))
        if key in predictions:
            del predictions[key]
        predictions[key] = str(repair["new_value"])
    matched = {key for key, value in predictions.items() if truth_map.get(key) == value}
    tp = len(matched)
    fp = sum(1 for key, value in predictions.items() if truth_map.get(key) != value)
    fn = len(set(truth_map) - matched)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return {
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "precision": round(precision, 4),
        "recall": round(recall, 4),
        "f1": round(f1, 4),
    }


def repair_failure_taxonomy(truth, repairs, allowed_columns, valid_rows):
    columns = set(allowed_columns)
    lowered = {column.lower() for column in columns}
    rows = {int(row) for row in valid_rows}
    truth_map = {(int(cell["row"]), str(cell["column"])): str(cell["clean_value"]) for cell in truth}
    predictions = {(int(repair["row"]), str(repair["column"])): str(repair["new_value"]) for repair in repairs}
    counts = Counter()
    for repair in repairs:
        row = int(repair["row"])
        column = str(repair["column"])
        key = (row, column)
        if column not in columns:
            counts["schema_case_error" if column.lower() in lowered else "wrong_cell"] += 1
            continue
        if row not in rows:
            counts["wrong_cell"] += 1
            continue
        if key not in truth_map:
            counts["overrepair"] += 1
            continue
        if truth_map[key] != str(repair["new_value"]):
            counts["wrong_value"] += 1
    for key in truth_map:
        if key not in predictions:
            counts["missed_repair"] += 1
    return {kind: counts[kind] for kind in sorted(counts) if counts[kind]}


def build_eval_tasks():
    target = int(config["evaluation"]["heldout_tasks"])
    datasets = {name: load_bench_dataset(name) for name in config["evaluation"]["datasets"]}
    tasks = []
    seed = int(config["evaluation"]["seeds_start"])
    attempts = 0
    max_attempts = max(target * 20, 20)
    while len(tasks) < target and attempts < max_attempts:
        for dataset_name, (dirty, columns, truth) in datasets.items():
            prefer_truth = attempts % 5 != 4
            rows = chunk_rows(dirty, columns, truth, seed + attempts, prefer_truth=prefer_truth)
            attempts += 1
            row_ids = {int(row["_row"]) for row in rows}
            task_truth = [cell for cell in truth if cell["row"] in row_ids]
            inferability = inferability_for_eval_task(dataset_name, rows, task_truth)
            tasks.append(
                {
                    "dataset": dataset_name,
                    "schema_summary": {"dataset": dataset_name, "columns": list(columns)},
                    "target_rows": rows,
                    "context_rows": [],
                    "allowed_columns": list(columns),
                    "valid_rows": [int(row["_row"]) for row in rows],
                    "ground_truth": task_truth,
                    "inferability": inferability,
                }
            )
            if len(tasks) >= target or attempts >= max_attempts:
                break
    if len(tasks) < target:
        raise RuntimeError(f"Could only build {len(tasks)} held-out tasks; requested {target}.")
    return tasks


def summarize_task_scores(task_scores):
    by_dataset = {}
    by_slice = {}
    failures = Counter()
    for row in task_scores:
        by_dataset.setdefault(row["dataset"], []).append(row["f1"])
        by_slice.setdefault(row["inferability"], []).append(row)
        failures.update(row.get("failure_taxonomy", {}))
        if not row["parse_ok"]:
            failures[row.get("parse_error_kind") or "parse_failure"] += 1
    dataset_f1 = {
        dataset: round(sum(values) / len(values), 4)
        for dataset, values in sorted(by_dataset.items())
        if values
    }
    macro_f1 = round(sum(dataset_f1.values()) / len(dataset_f1), 4) if dataset_f1 else 0.0
    mean_f1 = round(sum(row["f1"] for row in task_scores) / len(task_scores), 4) if task_scores else 0.0
    parse_success_rate = (
        round(sum(1 for row in task_scores if row["parse_ok"]) / len(task_scores), 4)
        if task_scores
        else 0.0
    )
    schema_case_error_count = sum(row["schema_case_errors"] for row in task_scores)
    slice_scores = {}
    for label, rows in sorted(by_slice.items()):
        slice_dataset = {}
        for row in rows:
            slice_dataset.setdefault(row["dataset"], []).append(row["f1"])
        slice_dataset_f1 = {
            dataset: round(sum(values) / len(values), 4)
            for dataset, values in sorted(slice_dataset.items())
            if values
        }
        predictions = sum(row["tp"] + row["fp"] for row in rows)
        finish_count = sum(1 for row in rows if row["tp"] + row["fp"] == 0)
        false_repair_count = sum(row["fp"] for row in rows)
        slice_scores[label] = {
            "tasks": len(rows),
            "macro_f1": round(sum(slice_dataset_f1.values()) / len(slice_dataset_f1), 4) if slice_dataset_f1 else 0.0,
            "mean_f1": round(sum(row["f1"] for row in rows) / len(rows), 4) if rows else 0.0,
            "parse_success_rate": round(sum(1 for row in rows if row["parse_ok"]) / len(rows), 4) if rows else 0.0,
            "schema_case_error_count": sum(row["schema_case_errors"] for row in rows),
            "finish_rate": round(finish_count / len(rows), 4) if rows else 0.0,
            "false_repair_rate": round(false_repair_count / predictions, 4) if predictions else 0.0,
            "false_repair_count": false_repair_count,
            "dataset_f1": slice_dataset_f1,
        }
    return {
        "macro_f1": macro_f1,
        "mean_f1": mean_f1,
        "dataset_f1": dataset_f1,
        "parse_success_rate": parse_success_rate,
        "schema_case_error_count": schema_case_error_count,
        "failure_taxonomy": {kind: failures[kind] for kind in sorted(failures)},
        "promotion_slice": PROMOTION_SLICE,
        "slice_scores": slice_scores,
    }


def build_prompt_messages(task):
    return [
        {"role": "system", "content": SYSTEM_PROMPT},
        {
            "role": "user",
            "content": json.dumps(
                {
                    "contract_version": config["collection"].get("prompt_contract_version", "repair_contract_v1"),
                    "schema_summary": task["schema_summary"],
                    "allowed_columns": task["allowed_columns"],
                    "valid_rows": task["valid_rows"],
                    "target_rows": task["target_rows"],
                    "context_rows": task["context_rows"],
                },
                sort_keys=True,
                separators=(",", ":"),
            ),
        },
    ]


def evaluate_model(eval_model, eval_tokenizer, tasks, *, model_label):
    eval_model.eval()
    device = next(eval_model.parameters()).device
    task_scores = []
    failure_samples = []
    failure_samples_by_slice = {label: [] for label in VALID_INFERABILITY_LABELS}
    max_failure_samples = int(config.get("release", {}).get("max_failure_samples", 25))
    for task_index, task in enumerate(tasks, start=1):
        dataset_name = task.get("dataset") or task.get("schema_summary", {}).get("dataset", "unknown")
        prompt_messages = build_prompt_messages(task)
        prompt = eval_tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
        inputs = eval_tokenizer(prompt, return_tensors="pt").to(device)
        with torch.inference_mode():
            output = eval_model.generate(
                **inputs,
                max_new_tokens=config["evaluation"]["max_new_tokens"],
                do_sample=False,
                pad_token_id=eval_tokenizer.eos_token_id,
            )
        decoded = eval_tokenizer.decode(output[0][inputs["input_ids"].shape[-1] :], skip_special_tokens=True)
        repairs, parse_error_kind = parse_repairs(decoded)
        score = score_repairs(task["ground_truth"], repairs)
        valid_rows = [int(row["_row"]) for row in task["target_rows"]]
        taxonomy = repair_failure_taxonomy(task["ground_truth"], repairs, task["allowed_columns"], valid_rows)
        schema_case_error_count = int(taxonomy.get("schema_case_error", 0))
        parse_ok = parse_error_kind is None
        row = {
            "task_index": task_index,
            "dataset": dataset_name,
            "inferability": task["inferability"],
            "f1": score["f1"],
            "precision": score["precision"],
            "recall": score["recall"],
            "tp": score["tp"],
            "fp": score["fp"],
            "fn": score["fn"],
            "parse_ok": parse_ok,
            "parse_error_kind": parse_error_kind,
            "schema_case_errors": schema_case_error_count,
            "failure_taxonomy": taxonomy,
        }
        task_scores.append(row)
        if score["f1"] < 1.0 or taxonomy or not parse_ok:
            sample = {
                **row,
                "target_rows": task["target_rows"],
                "ground_truth": task["ground_truth"],
                "predicted_repairs": repairs[:20],
                "decoded_preview": decoded[:1500],
            }
            if len(failure_samples) < max_failure_samples:
                failure_samples.append(sample)
            slice_samples = failure_samples_by_slice.setdefault(task["inferability"], [])
            if len(slice_samples) < max_failure_samples:
                slice_samples.append(sample)
        if task_index % 10 == 0:
            print(f"{model_label}: evaluated {task_index}/{len(tasks)} tasks", flush=True)
    summary = summarize_task_scores(task_scores)
    diagnostics = {
        "model_label": model_label,
        "task_scores": task_scores,
        "failure_samples": failure_samples,
        "failure_samples_by_slice": {label: samples for label, samples in failure_samples_by_slice.items() if samples},
    }
    return summary, diagnostics


def evaluate_checkpoint(model_ref, *, model_label, token=None):
    eval_model = AutoModelForCausalLM.from_pretrained(
        model_ref,
        dtype=torch.float16,
        device_map="auto",
        token=token,
        trust_remote_code=config["model"]["trust_remote_code"],
    )
    scores, diagnostics = evaluate_model(eval_model, tokenizer, tasks, model_label=model_label)
    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
    return scores, diagnostics


def promotion_summary(summary):
    return summary.get("slice_scores", {}).get(PROMOTION_SLICE, summary)


def quality_gate_failures(base_eval, sft_eval):
    failures = []
    base_promotion = promotion_summary(base_eval)
    sft_promotion = promotion_summary(sft_eval)
    if float(sft_promotion["macro_f1"]) <= float(base_promotion["macro_f1"]):
        failures.append("sft_f1>base_f1")
    if float(sft_promotion.get("parse_success_rate", 0.0)) < float(config["evaluation"].get("parse_success_min", 0.99)):
        failures.append("parse_success_rate>=0.99")
    if int(sft_promotion.get("schema_case_error_count", 0)) > int(config["evaluation"].get("schema_case_error_max", 0)):
        failures.append("schema_case_error_count==0")
    return failures


tasks = build_eval_tasks()
base_eval, base_diagnostics = evaluate_checkpoint(base_model_id, model_label="base", token=HF_TOKEN)
sft_eval, sft_diagnostics = evaluate_checkpoint(merged_dir, model_label="sft")
base_promotion_eval = promotion_summary(base_eval)
sft_promotion_eval = promotion_summary(sft_eval)
base_f1 = base_promotion_eval["macro_f1"]
sft_f1 = sft_promotion_eval["macro_f1"]
parse_success_rate = sft_promotion_eval.get("parse_success_rate", 0.0)
schema_case_error_count = int(sft_promotion_eval.get("schema_case_error_count", 0))
prompt_contract_drift = False
heldout_leakage_detected = False
gate_failures = quality_gate_failures(base_eval, sft_eval)
if prompt_contract_drift:
    gate_failures.append("prompt_contract_drift==false")
if heldout_leakage_detected:
    gate_failures.append("heldout_leakage_detected==false")
quality_milestone = not gate_failures
release_status = (
    "quality_improved_verified"
    if quality_milestone
    else ("diagnostic_complete_no_gain" if sft_f1 <= base_f1 else "quality_gate_failed")
)
training_metrics = {
    "model_name": "DataForge-0.5B-SFT",
    "model_license": config["model"]["model_license"],
    "base_model": base_model_id,
    "teacher_model": config["collection"]["teacher_model"],
    "dataset_repo": DATASET_REPO,
    "dataset_sha": DATASET_SHA,
    "dataset_records": len(raw_dataset),
    "training_examples": len(train_records),
    "heldout_examples": len(heldout_records),
    "trajectory_filename": config["repos"]["trajectory_filename"],
    "promotion_slice": PROMOTION_SLICE,
    "config_sha256": file_sha256(CONFIG_PATH),
    "trajectory_sha256": file_sha256(DATA_PATH),
    "split_manifest_sha256": file_sha256(SPLIT_MANIFEST_PATH),
    "kaggle_hours": round((time.time() - globals().get("_START_TIME", time.time())) / 3600, 3),
    "base_f1": base_f1,
    "sft_f1": sft_f1,
    "base_eval": base_eval,
    "sft_eval": sft_eval,
    "slice_scores": {"base": base_eval.get("slice_scores", {}), "sft": sft_eval.get("slice_scores", {})},
    "evaluation_chunk_width": int(config["evaluation"].get("chunk_width", 4)),
    "evaluation_max_new_tokens": int(config["evaluation"].get("max_new_tokens", 1024)),
    "parse_success_rate": parse_success_rate,
    "schema_case_error_count": schema_case_error_count,
    "quality_gate_failures": gate_failures,
    "prompt_contract_drift": prompt_contract_drift,
    "heldout_leakage_detected": heldout_leakage_detected,
    "quality_milestone": quality_milestone,
    "release_status": release_status,
    "repo_id": MODEL_REPO,
}
eval_diagnostics = {
    "schema_version": "dataforge_eval_diagnostics_v1",
    "model_repo": MODEL_REPO,
    "dataset_repo": DATASET_REPO,
    "dataset_sha": DATASET_SHA,
    "base_model": base_model_id,
    "quality_gate_failures": gate_failures,
    "promotion_slice": PROMOTION_SLICE,
    "slice_scores": {"base": base_eval.get("slice_scores", {}), "sft": sft_eval.get("slice_scores", {})},
    "base": base_diagnostics,
    "sft": sft_diagnostics,
}
(merged_dir / "training_metrics.json").write_text(json.dumps(training_metrics, indent=2), encoding="utf-8")
(merged_dir / "eval_diagnostics.json").write_text(json.dumps(eval_diagnostics, indent=2), encoding="utf-8")
(merged_dir / "README.md").write_text(
    CARD_TEMPLATE_PATH.read_text(encoding="utf-8").format(**training_metrics), encoding="utf-8"
)
api.create_repo(repo_id=MODEL_REPO, repo_type="model", exist_ok=True, token=HF_TOKEN)
api.upload_folder(
    folder_path=str(merged_dir),
    repo_id=MODEL_REPO,
    repo_type="model",
    token=HF_TOKEN,
    commit_message=f"Publish DataForge 0.5B SFT {config['collection']['schema_version']} ({release_status}) from dataset {DATASET_SHA[:8]}",
)
print(f"Base promotion F1 ({PROMOTION_SLICE}): {base_f1}")
print(f"SFT promotion F1 ({PROMOTION_SLICE}): {sft_f1}")
print(f"Release status: {training_metrics['release_status']}")
print(f"Quality gate failures: {gate_failures}")
print(f"Pushed merged checkpoint and diagnostics to {MODEL_REPO}")
